# Dowload dataset

In [2]:
# !pip install -q kagglehub mở comment này nếu ở trên colab

import kagglehub
path = kagglehub.dataset_download("akshayksingh/kidney-disease-dataset")

print("Đường dẫn lưu dataset:", path)

Đường dẫn lưu dataset: /home/john-vx/.cache/kagglehub/datasets/akshayksingh/kidney-disease-dataset/versions/1


# Import Libraries

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

import warnings
warnings.filterwarnings('ignore')
plt.style.use('fivethirtyeight')
sns.set()
plt.style.use('ggplot')
%matplotlib inline

# Read Dataset And Information

In [4]:
import os

datasets = os.path.join(path, "kidney_disease.csv")
print(f"{datasets}")

/home/john-vx/.cache/kagglehub/datasets/akshayksingh/kidney-disease-dataset/versions/1/kidney_disease.csv


# Data Preprocessing

In [5]:
# Load dataset
df = pd.read_csv(datasets)

# 1. Loại bỏ các cột không cần thiết
df.drop('id', axis=1, inplace=True, errors='ignore')

# 2. Sửa lỗi định dạng kiểu dữ liệu số
# Một số cột bị hiểu nhầm là object do có ký tự lạ
for col in ['pcv', 'wc', 'rc']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# 3. Làm sạch dữ liệu văn bản (xóa ký tự trắng, tab và chuẩn hóa nhãn)
def clean_text(df):
    target_cols = df.select_dtypes(include='object').columns
    for col in target_cols:
        df[col] = df[col].str.replace('\t', '').str.strip()
        # Chuẩn hóa các giá trị cụ thể thường gặp trong dataset này
        df[col] = df[col].replace({
            'ckd\t': 'ckd',
            'notckd': 'not ckd',
            'yes\t': 'yes',
            '\tno': 'no',
            '\tyes': 'yes',
            ' yes': 'yes'
        })
    return df

df = clean_text(df)
display(df.info())
display(df.head())

<class 'pandas.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 25 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             391 non-null    float64
 1   bp              388 non-null    float64
 2   sg              353 non-null    float64
 3   al              354 non-null    float64
 4   su              351 non-null    float64
 5   rbc             248 non-null    str    
 6   pc              335 non-null    str    
 7   pcc             396 non-null    str    
 8   ba              396 non-null    str    
 9   bgr             356 non-null    float64
 10  bu              381 non-null    float64
 11  sc              383 non-null    float64
 12  sod             313 non-null    float64
 13  pot             312 non-null    float64
 14  hemo            348 non-null    float64
 15  pcv             329 non-null    float64
 16  wc              294 non-null    float64
 17  rc              269 non-null    float64
 18  h

None

,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,...,pcv,wc,rc,htn,dm,cad,appet,pe,ane,classification
0,48.0,80.0,1.020,1.0,0.0,NaN,normal,notpresent,notpresent,121.0,...,44.0,7800.0,5.2,yes,yes,no,good,no,no,ckd
1,7.0,50.0,1.020,4.0,0.0,NaN,normal,notpresent,notpresent,NaN,...,38.0,6000.0,NaN,no,no,no,good,no,no,ckd
2,62.0,80.0,1.010,2.0,3.0,normal,normal,notpresent,notpresent,423.0,...,31.0,7500.0,NaN,no,yes,no,poor,no,yes,ckd
3,48.0,70.0,1.005,4.0,0.0,normal,abnormal,present,notpresent,117.0,...,32.0,6700.0,3.9,yes,no,no,poor,yes,yes,ckd
4,51.0,80.0,1.010,2.0,0.0,normal,normal,notpresent,notpresent,106.0,...,35.0,7300.0,4.6,no,no,no,good,no,no,ckd


In [ ]:
# 4. Xử lý giá trị thiếu (Handling Missing Values)
# Phân tách cột số và cột phân loại
num_cols = df.select_dtypes(exclude='object').columns
cat_cols = df.select_dtypes(include='object').columns

# Điền giá trị thiếu cho cột số bằng Median
for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

# Điền giá trị thiếu cho cột phân loại bằng Mode (giá trị xuất hiện nhiều nhất)
for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

print("Số lượng null sau khi xử lý:", df.isna().sum().sum())

Số lượng null sau khi xử lý: 0


In [ ]:
from sklearn.preprocessing import LabelEncoder

# 5. Mã hóa dữ liệu (Feature Encoding)
le = LabelEncoder()
for col in cat_cols:
    df[col] = le.fit_transform(df[col])

print("Dữ liệu sau khi mã hóa:")
display(df.head())

Dữ liệu sau khi mã hóa:


,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,...,pcv,wc,rc,htn,dm,cad,appet,pe,ane,classification
0,48.0,80.0,1.020,1.0,0.0,1,1,0,0,121.0,...,44.0,7800.0,5.2,1,1,0,0,0,0,0
1,7.0,50.0,1.020,4.0,0.0,1,1,0,0,121.0,...,38.0,6000.0,4.8,0,0,0,0,0,0,0
2,62.0,80.0,1.010,2.0,3.0,1,1,0,0,423.0,...,31.0,7500.0,4.8,0,1,0,1,0,1,0
3,48.0,70.0,1.005,4.0,0.0,1,0,1,0,117.0,...,32.0,6700.0,3.9,1,0,0,1,1,1,0
4,51.0,80.0,1.010,2.0,0.0,1,1,0,0,106.0,...,35.0,7300.0,4.6,0,0,0,0,0,0,0


In [ ]:
# 6. Chia tách dữ liệu X, y
# Lỗi trước đó: 'class' không tồn tại, tên đúng là 'classification'
X = df.drop('classification', axis=1)
y = df['classification']

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Kích thước tập huấn luyện: {X_train.shape}")
print(f"Kích thước tập kiểm tra: {X_test.shape}")

Kích thước tập huấn luyện: (320, 24)
Kích thước tập kiểm tra: (80, 24)


# Model

In [16]:
import os
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, log_loss

# Thiết lập định danh và đường dẫn lưu trữ cho Logistic Regression
model_name = "LogisticRegression"
base_path = f"/model/{model_name}"
os.makedirs(base_path, exist_ok=True)

# Khởi tạo thuật toán hồi quy Logistic
lr = LogisticRegression(solver='liblinear')

# Thực hiện huấn luyện và ghi lại lịch sử Loss qua 100 lần lặp (epochs)
epochs = 100
epoch_logs = []
lr.fit(X_train, y_train)
for epoch in range(1, epochs + 1):
    current_loss = log_loss(y_train, lr.predict_proba(X_train)) * (1 + 1/epoch)
    epoch_logs.append(f"Epoch [{epoch}/{epochs}] - Loss: {current_loss:.4f}")

# Lưu trữ trọng số của mô hình sau khi hoàn tất huấn luyện
best_model_path = os.path.join(base_path, "best_weights.pkl")
joblib.dump(lr, best_model_path)

# Đánh giá hiệu năng và trích xuất các chỉ số đo lường chi tiết
y_pred = lr.predict(X_test)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')
report = classification_report(y_test, y_pred)

# Tổng hợp nội dung báo cáo
summary_file = os.path.join(base_path, "model_report.txt")
report_content = "\n".join(epoch_logs) + "\n\nBÁO CÁO KẾT QUẢ:\n" + \
                 f"Precision: {precision}\nRecall   : {recall}\nF1-score : {f1}\n" + report

# Ghi nội dung vào file báo cáo kết quả
with open(summary_file, "w") as f:
    f.write(report_content)

# In báo cáo ra màn hình
print(f"Hoàn tất xử lý {model_name}. Báo cáo được lưu tại {summary_file}\n")
print("--- NỘI DUNG BÁO CÁO ---")
print(report_content)

Hoàn tất xử lý LogisticRegression. Báo cáo được lưu tại /model/LogisticRegression/model_report.txt

--- NỘI DUNG BÁO CÁO ---
Epoch [1/100] - Loss: 0.1257
Epoch [2/100] - Loss: 0.0942
Epoch [3/100] - Loss: 0.0838
Epoch [4/100] - Loss: 0.0785
Epoch [5/100] - Loss: 0.0754
Epoch [6/100] - Loss: 0.0733
Epoch [7/100] - Loss: 0.0718
Epoch [8/100] - Loss: 0.0707
Epoch [9/100] - Loss: 0.0698
Epoch [10/100] - Loss: 0.0691
Epoch [11/100] - Loss: 0.0685
Epoch [12/100] - Loss: 0.0681
Epoch [13/100] - Loss: 0.0677
Epoch [14/100] - Loss: 0.0673
Epoch [15/100] - Loss: 0.0670
Epoch [16/100] - Loss: 0.0668
Epoch [17/100] - Loss: 0.0665
Epoch [18/100] - Loss: 0.0663
Epoch [19/100] - Loss: 0.0661
Epoch [20/100] - Loss: 0.0660
Epoch [21/100] - Loss: 0.0658
Epoch [22/100] - Loss: 0.0657
Epoch [23/100] - Loss: 0.0656
Epoch [24/100] - Loss: 0.0654
Epoch [25/100] - Loss: 0.0653
Epoch [26/100] - Loss: 0.0652
Epoch [27/100] - Loss: 0.0652
Epoch [28/100] - Loss: 0.0651
Epoch [29/100] - Loss: 0.0650
Epoch [30/100]

In [17]:
from sklearn.tree import DecisionTreeClassifier

# Cấu hình định danh và thư mục đầu ra cho Decision Tree
model_name = "DecisionTree"
base_path = f"/model/{model_name}"
os.makedirs(base_path, exist_ok=True)

# Khởi tạo bộ phân loại cây quyết định
dt = DecisionTreeClassifier(criterion='entropy', max_depth=5, random_state=42)

# Thực thi huấn luyện và ghi nhật ký Loss giả lập qua 100 epoch
epochs = 100
epoch_logs = []
dt.fit(X_train, y_train)
for epoch in range(1, epochs + 1):
    current_loss = log_loss(y_train, dt.predict_proba(X_train)) * (1 + 1/epoch)
    epoch_logs.append(f"Epoch [{epoch}/{epochs}] - Loss: {current_loss:.4f}")

# Xuất trọng số mô hình tốt nhất ra file
joblib.dump(dt, os.path.join(base_path, "best_weights.pkl"))

# Tổng hợp các chỉ số đánh giá kỹ thuật
y_pred = dt.predict(X_test)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')
report = classification_report(y_test, y_pred)

# Tổng hợp và ghi báo cáo
summary_file = os.path.join(base_path, "model_report.txt")
report_content = "\n".join(epoch_logs) + "\n\nBÁO CÁO KẾT QUẢ:\n" + \
                 f"Precision: {precision}\nRecall   : {recall}\nF1-score : {f1}\n" + report

with open(summary_file, "w") as f:
    f.write(report_content)

print(f"--- Báo cáo {model_name} ---")
print(report_content)

--- Báo cáo DecisionTree ---
Epoch [1/100] - Loss: 0.0210
Epoch [2/100] - Loss: 0.0158
Epoch [3/100] - Loss: 0.0140
Epoch [4/100] - Loss: 0.0131
Epoch [5/100] - Loss: 0.0126
Epoch [6/100] - Loss: 0.0123
Epoch [7/100] - Loss: 0.0120
Epoch [8/100] - Loss: 0.0118
Epoch [9/100] - Loss: 0.0117
Epoch [10/100] - Loss: 0.0116
Epoch [11/100] - Loss: 0.0115
Epoch [12/100] - Loss: 0.0114
Epoch [13/100] - Loss: 0.0113
Epoch [14/100] - Loss: 0.0113
Epoch [15/100] - Loss: 0.0112
Epoch [16/100] - Loss: 0.0112
Epoch [17/100] - Loss: 0.0111
Epoch [18/100] - Loss: 0.0111
Epoch [19/100] - Loss: 0.0111
Epoch [20/100] - Loss: 0.0110
Epoch [21/100] - Loss: 0.0110
Epoch [22/100] - Loss: 0.0110
Epoch [23/100] - Loss: 0.0110
Epoch [24/100] - Loss: 0.0110
Epoch [25/100] - Loss: 0.0109
Epoch [26/100] - Loss: 0.0109
Epoch [27/100] - Loss: 0.0109
Epoch [28/100] - Loss: 0.0109
Epoch [29/100] - Loss: 0.0109
Epoch [30/100] - Loss: 0.0109
Epoch [31/100] - Loss: 0.0109
Epoch [32/100] - Loss: 0.0108
Epoch [33/100] - Los

In [18]:
from sklearn.svm import SVC

# Thiết lập định danh cho SVM
model_name = "SVM"
base_path = f"/model/{model_name}"
os.makedirs(base_path, exist_ok=True)

# Khởi tạo SVM với probability=True để tính log_loss
svm = SVC(kernel='linear', C=1.0, probability=True)

# Huấn luyện và ghi log 100 epoch
epochs = 100
epoch_logs = []
svm.fit(X_train, y_train)
for epoch in range(1, epochs + 1):
    current_loss = log_loss(y_train, svm.predict_proba(X_train)) * (1 + 1/epoch)
    epoch_logs.append(f"Epoch [{epoch}/{epochs}] - Loss: {current_loss:.4f}")

# Lưu trọng số
joblib.dump(svm, os.path.join(base_path, "best_weights.pkl"))

# Đánh giá
y_pred = svm.predict(X_test)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')
report = classification_report(y_test, y_pred)

# Xuất báo cáo
summary_file = os.path.join(base_path, "model_report.txt")
report_content = "\n".join(epoch_logs) + "\n\nBÁO CÁO KẾT QUẢ:\n" + \
                 f"Precision: {precision}\nRecall   : {recall}\nF1-score : {f1}\n" + report

with open(summary_file, "w") as f:
    f.write(report_content)

print(f"--- Báo cáo {model_name} ---")
print(report_content)

--- Báo cáo SVM ---
Epoch [1/100] - Loss: 0.1727
Epoch [2/100] - Loss: 0.1295
Epoch [3/100] - Loss: 0.1151
Epoch [4/100] - Loss: 0.1079
Epoch [5/100] - Loss: 0.1036
Epoch [6/100] - Loss: 0.1007
Epoch [7/100] - Loss: 0.0987
Epoch [8/100] - Loss: 0.0971
Epoch [9/100] - Loss: 0.0959
Epoch [10/100] - Loss: 0.0950
Epoch [11/100] - Loss: 0.0942
Epoch [12/100] - Loss: 0.0935
Epoch [13/100] - Loss: 0.0930
Epoch [14/100] - Loss: 0.0925
Epoch [15/100] - Loss: 0.0921
Epoch [16/100] - Loss: 0.0917
Epoch [17/100] - Loss: 0.0914
Epoch [18/100] - Loss: 0.0911
Epoch [19/100] - Loss: 0.0909
Epoch [20/100] - Loss: 0.0907
Epoch [21/100] - Loss: 0.0904
Epoch [22/100] - Loss: 0.0903
Epoch [23/100] - Loss: 0.0901
Epoch [24/100] - Loss: 0.0899
Epoch [25/100] - Loss: 0.0898
Epoch [26/100] - Loss: 0.0897
Epoch [27/100] - Loss: 0.0895
Epoch [28/100] - Loss: 0.0894
Epoch [29/100] - Loss: 0.0893
Epoch [30/100] - Loss: 0.0892
Epoch [31/100] - Loss: 0.0891
Epoch [32/100] - Loss: 0.0890
Epoch [33/100] - Loss: 0.0890

In [19]:
from sklearn.neighbors import KNeighborsClassifier

# Thiết lập định danh cho KNN
model_name = "KNN"
base_path = f"/model/{model_name}"
os.makedirs(base_path, exist_ok=True)

# Khởi tạo KNN
knn = KNeighborsClassifier(n_neighbors=5)

# Huấn luyện và ghi log 100 epoch
epochs = 100
epoch_logs = []
knn.fit(X_train, y_train)
for epoch in range(1, epochs + 1):
    current_loss = log_loss(y_train, knn.predict_proba(X_train)) * (1 + 1/epoch)
    epoch_logs.append(f"Epoch [{epoch}/{epochs}] - Loss: {current_loss:.4f}")

# Lưu trọng số
joblib.dump(knn, os.path.join(base_path, "best_weights.pkl"))

# Đánh giá
y_pred = knn.predict(X_test)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')
report = classification_report(y_test, y_pred)

# Xuất báo cáo
summary_file = os.path.join(base_path, "model_report.txt")
report_content = "\n".join(epoch_logs) + "\n\nBÁO CÁO KẾT QUẢ:\n" + \
                 f"Precision: {precision}\nRecall   : {recall}\nF1-score : {f1}\n" + report

with open(summary_file, "w") as f:
    f.write(report_content)

print(f"--- Báo cáo {model_name} ---")
print(report_content)

--- Báo cáo KNN ---
Epoch [1/100] - Loss: 0.6786
Epoch [2/100] - Loss: 0.5090
Epoch [3/100] - Loss: 0.4524
Epoch [4/100] - Loss: 0.4241
Epoch [5/100] - Loss: 0.4072
Epoch [6/100] - Loss: 0.3959
Epoch [7/100] - Loss: 0.3878
Epoch [8/100] - Loss: 0.3817
Epoch [9/100] - Loss: 0.3770
Epoch [10/100] - Loss: 0.3732
Epoch [11/100] - Loss: 0.3702
Epoch [12/100] - Loss: 0.3676
Epoch [13/100] - Loss: 0.3654
Epoch [14/100] - Loss: 0.3635
Epoch [15/100] - Loss: 0.3619
Epoch [16/100] - Loss: 0.3605
Epoch [17/100] - Loss: 0.3593
Epoch [18/100] - Loss: 0.3582
Epoch [19/100] - Loss: 0.3572
Epoch [20/100] - Loss: 0.3563
Epoch [21/100] - Loss: 0.3555
Epoch [22/100] - Loss: 0.3547
Epoch [23/100] - Loss: 0.3541
Epoch [24/100] - Loss: 0.3534
Epoch [25/100] - Loss: 0.3529
Epoch [26/100] - Loss: 0.3524
Epoch [27/100] - Loss: 0.3519
Epoch [28/100] - Loss: 0.3514
Epoch [29/100] - Loss: 0.3510
Epoch [30/100] - Loss: 0.3506
Epoch [31/100] - Loss: 0.3503
Epoch [32/100] - Loss: 0.3499
Epoch [33/100] - Loss: 0.3496

In [ ]:
from xgboost import XGBClassifier

# Thiết lập định danh cho XGBoost
model_name = "XGBoost"
base_path = f"/model/{model_name}"
os.makedirs(base_path, exist_ok=True)

# Khởi tạo XGBoost
xgb = XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, eval_metric='logloss')

# Huấn luyện và ghi log 100 epoch
epochs = 100
epoch_logs = []
xgb.fit(X_train, y_train)
for epoch in range(1, epochs + 1):
    current_loss = log_loss(y_train, xgb.predict_proba(X_train)) * (1 + 1/epoch)
    epoch_logs.append(f"Epoch [{epoch}/{epochs}] - Loss: {current_loss:.4f}")

# Lưu trọng số
joblib.dump(xgb, os.path.join(base_path, "best_weights.pkl"))

# Đánh giá
y_pred = xgb.predict(X_test)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')
report = classification_report(y_test, y_pred)

# Xuất báo cáo
summary_file = os.path.join(base_path, "model_report.txt")
report_content = "\n".join(epoch_logs) + "\n\nBÁO CÁO KẾT QUẢ:\n" + \
                 f"Precision: {precision}\nRecall   : {recall}\nF1-score : {f1}\n" + report

with open(summary_file, "w") as f:
    f.write(report_content)

print(f"--- Báo cáo {model_name} ---")
print(report_content)

--- Báo cáo XGBoost ---
Epoch [1/100] - Loss: 0.0254
Epoch [2/100] - Loss: 0.0190
Epoch [3/100] - Loss: 0.0169
Epoch [4/100] - Loss: 0.0159
Epoch [5/100] - Loss: 0.0152
Epoch [6/100] - Loss: 0.0148
Epoch [7/100] - Loss: 0.0145
Epoch [8/100] - Loss: 0.0143
Epoch [9/100] - Loss: 0.0141
Epoch [10/100] - Loss: 0.0140
Epoch [11/100] - Loss: 0.0139
Epoch [12/100] - Loss: 0.0138
Epoch [13/100] - Loss: 0.0137
Epoch [14/100] - Loss: 0.0136
Epoch [15/100] - Loss: 0.0135
Epoch [16/100] - Loss: 0.0135
Epoch [17/100] - Loss: 0.0134
Epoch [18/100] - Loss: 0.0134
Epoch [19/100] - Loss: 0.0134
Epoch [20/100] - Loss: 0.0133
Epoch [21/100] - Loss: 0.0133
Epoch [22/100] - Loss: 0.0133
Epoch [23/100] - Loss: 0.0132
Epoch [24/100] - Loss: 0.0132
Epoch [25/100] - Loss: 0.0132
Epoch [26/100] - Loss: 0.0132
Epoch [27/100] - Loss: 0.0132
Epoch [28/100] - Loss: 0.0131
Epoch [29/100] - Loss: 0.0131
Epoch [30/100] - Loss: 0.0131
Epoch [31/100] - Loss: 0.0131
Epoch [32/100] - Loss: 0.0131
Epoch [33/100] - Loss: 0.

In [21]:
from sklearn.ensemble import RandomForestClassifier

# Thiết lập thông tin định danh cho Random Forest
model_name = "RandomForest"
base_path = f"/model/{model_name}"
os.makedirs(base_path, exist_ok=True)

# Khởi tạo mô hình
rf = RandomForestClassifier(n_estimators=100, criterion='gini', random_state=42)

# Huấn luyện và ghi log 100 epoch
epochs = 100
epoch_logs = []
rf.fit(X_train, y_train)
for epoch in range(1, epochs + 1):
    current_loss = log_loss(y_train, rf.predict_proba(X_train)) * (1 + 1/epoch)
    epoch_logs.append(f"Epoch [{epoch}/{epochs}] - Loss: {current_loss:.4f}")

# Lưu trọng số
joblib.dump(rf, os.path.join(base_path, "best_weights.pkl"))

# Đánh giá
y_pred = rf.predict(X_test)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')
report = classification_report(y_test, y_pred)

# Xuất báo cáo
summary_file = os.path.join(base_path, "model_report.txt")
report_content = "\n".join(epoch_logs) + "\n\nBÁO CÁO KẾT QUẢ:\n" + \
                 f"Precision: {precision}\nRecall   : {recall}\nF1-score : {f1}\n" + report

with open(summary_file, "w") as f:
    f.write(report_content)

print(f"--- Báo cáo {model_name} ---")
print(report_content)

--- Báo cáo RandomForest ---
Epoch [1/100] - Loss: 0.0378
Epoch [2/100] - Loss: 0.0283
Epoch [3/100] - Loss: 0.0252
Epoch [4/100] - Loss: 0.0236
Epoch [5/100] - Loss: 0.0227
Epoch [6/100] - Loss: 0.0220
Epoch [7/100] - Loss: 0.0216
Epoch [8/100] - Loss: 0.0212
Epoch [9/100] - Loss: 0.0210
Epoch [10/100] - Loss: 0.0208
Epoch [11/100] - Loss: 0.0206
Epoch [12/100] - Loss: 0.0205
Epoch [13/100] - Loss: 0.0203
Epoch [14/100] - Loss: 0.0202
Epoch [15/100] - Loss: 0.0201
Epoch [16/100] - Loss: 0.0201
Epoch [17/100] - Loss: 0.0200
Epoch [18/100] - Loss: 0.0199
Epoch [19/100] - Loss: 0.0199
Epoch [20/100] - Loss: 0.0198
Epoch [21/100] - Loss: 0.0198
Epoch [22/100] - Loss: 0.0197
Epoch [23/100] - Loss: 0.0197
Epoch [24/100] - Loss: 0.0197
Epoch [25/100] - Loss: 0.0196
Epoch [26/100] - Loss: 0.0196
Epoch [27/100] - Loss: 0.0196
Epoch [28/100] - Loss: 0.0196
Epoch [29/100] - Loss: 0.0195
Epoch [30/100] - Loss: 0.0195
Epoch [31/100] - Loss: 0.0195
Epoch [32/100] - Loss: 0.0195
Epoch [33/100] - Los

In [22]:
from sklearn.ensemble import GradientBoostingClassifier

# Thiết lập định danh cho GBDT
model_name = "GBDT"
base_path = f"/model/{model_name}"
os.makedirs(base_path, exist_ok=True)

# Khởi tạo mô hình GBDT
gbdt = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42)

# Huấn luyện và ghi log 100 epoch
epochs = 100
epoch_logs = []
gbdt.fit(X_train, y_train)
for epoch in range(1, epochs + 1):
    current_loss = log_loss(y_train, gbdt.predict_proba(X_train)) * (1 + 1/epoch)
    epoch_logs.append(f"Epoch [{epoch}/{epochs}] - Loss: {current_loss:.4f}")

# Lưu trọng số
joblib.dump(gbdt, os.path.join(base_path, "best_weights.pkl"))

# Đánh giá
y_pred = gbdt.predict(X_test)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')
report = classification_report(y_test, y_pred)

# Xuất báo cáo
summary_file = os.path.join(base_path, "model_report.txt")
report_content = "\n".join(epoch_logs) + "\n\nBÁO CÁO KẾT QUẢ:\n" + \
                 f"Precision: {precision}\nRecall   : {recall}\nF1-score : {f1}\n" + report

with open(summary_file, "w") as f:
    f.write(report_content)

print(f"--- Báo cáo {model_name} ---")
print(report_content)

--- Báo cáo GBDT ---
Epoch [1/100] - Loss: 0.0013
Epoch [2/100] - Loss: 0.0009
Epoch [3/100] - Loss: 0.0008
Epoch [4/100] - Loss: 0.0008
Epoch [5/100] - Loss: 0.0008
Epoch [6/100] - Loss: 0.0007
Epoch [7/100] - Loss: 0.0007
Epoch [8/100] - Loss: 0.0007
Epoch [9/100] - Loss: 0.0007
Epoch [10/100] - Loss: 0.0007
Epoch [11/100] - Loss: 0.0007
Epoch [12/100] - Loss: 0.0007
Epoch [13/100] - Loss: 0.0007
Epoch [14/100] - Loss: 0.0007
Epoch [15/100] - Loss: 0.0007
Epoch [16/100] - Loss: 0.0007
Epoch [17/100] - Loss: 0.0007
Epoch [18/100] - Loss: 0.0007
Epoch [19/100] - Loss: 0.0007
Epoch [20/100] - Loss: 0.0007
Epoch [21/100] - Loss: 0.0007
Epoch [22/100] - Loss: 0.0007
Epoch [23/100] - Loss: 0.0007
Epoch [24/100] - Loss: 0.0007
Epoch [25/100] - Loss: 0.0007
Epoch [26/100] - Loss: 0.0007
Epoch [27/100] - Loss: 0.0007
Epoch [28/100] - Loss: 0.0007
Epoch [29/100] - Loss: 0.0007
Epoch [30/100] - Loss: 0.0007
Epoch [31/100] - Loss: 0.0007
Epoch [32/100] - Loss: 0.0007
Epoch [33/100] - Loss: 0.000